# **CLEANING CSV: grocery_store_dataset_raw**

## **Load + View Dataset**

In [32]:
# Import libraries
import pandas as pd
import numpy as np
import re
from collections import Counter

# Load dataset
df = pd.read_csv("grocery_store_dataset_raw.csv")

In [33]:
# Inspect dataset
print("Dataset Dimensions: ", df.shape)
print("Dataset Attributes: ", df.columns.values)

df.info()
df.head(10)

Dataset Dimensions:  (1757, 8)
Dataset Attributes:  ['Sub Category' 'Price' 'Discount' 'Rating' 'Title' 'Currency' 'Feature'
 'Product Description']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1757 entries, 0 to 1756
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Sub Category         1757 non-null   object
 1   Price                1754 non-null   object
 2   Discount             1757 non-null   object
 3   Rating               682 non-null    object
 4   Title                1757 non-null   object
 5   Currency             1752 non-null   object
 6   Feature              1739 non-null   object
 7   Product Description  1715 non-null   object
dtypes: object(8)
memory usage: 109.9+ KB


,Sub Category,Price,Discount,Rating,Title,Currency,Feature,Product Description
0,Bakery & Desserts,$56.99,No Discount,Rated 4.3 out of 5 stars based on 265 reviews.,"David’s Cookies Mile High Peanut Butter Cake, ...",$,"""10"""" Peanut Butter Cake\nCertified Kosher OU-...",A cake the dessert epicure will die for!Our To...
1,Bakery & Desserts,$159.99,No Discount,Rated 5 out of 5 stars based on 1 reviews.,"The Cake Bake Shop 8"" Round Carrot Cake (16-22...",$,Spiced Carrot Cake with Cream Cheese Frosting ...,"Due to the perishable nature of this item, ord..."
2,Bakery & Desserts,$44.99,No Discount,Rated 4.1 out of 5 stars based on 441 reviews.,"St Michel Madeleine, Classic French Sponge Cak...",$,100 count\nIndividually wrapped\nMade in and I...,Moist and buttery sponge cakes with the tradit...
3,Bakery & Desserts,$39.99,No Discount,Rated 4.7 out of 5 stars based on 9459 reviews.,"David's Cookies Butter Pecan Meltaways 32 oz, ...",$,Butter Pecan Meltaways\n32 oz 2-Pack\nNo Prese...,These delectable butter pecan meltaways are th...
4,Bakery & Desserts,$59.99,No Discount,Rated 4.5 out of 5 stars based on 758 reviews.,"David’s Cookies Premier Chocolate Cake, 7.2 lb...",$,"""10"" Four Layer Chocolate Cake\nCertified Kosh...",A cake the dessert epicure will die for!To the...
5,Bakery & Desserts,$59.99,No Discount,Rated 4.4 out of 5 stars based on 369 reviews.,David's Cookies Mango & Strawberry Cheesecake ...,$,2-count\nStrawberry Cheesecake\nMango Cheeseca...,Strawberry Cheesecake: There's only one way to...
6,Bakery & Desserts,$74.99,No Discount,Rated 4.7 out of 5 stars based on 2241 reviews.,"La Grande Galette French Butter Cookies, 1.3 l...",$,"1.3 lb, 6-count\nBaked in, and Imported from, ...",Once upon a time in the French coastal town of...
7,Bakery & Desserts,$59.99,No Discount,Rated 4.4 out of 5 stars based on 232 reviews.,David's Cookies No Sugar Added Cheesecake & Ma...,$,2-count\nNo Sugar Added\nKosher OU-Dairy,Creamy Dreamy:This smooth creamy cheesecake ha...
8,Bakery & Desserts,$29.99,No Discount,Rated 4.4 out of 5 stars based on 1679 reviews.,David's Cookies Brownie and Cookie Combo Pack,$,6 Rocky Road Brownies\n12 Chocoloate Chunk Coo...,Due to the perishable nature of this product o...
9,Bakery & Desserts,$159.99,No Discount,Rated 5 out of 5 stars based on 2 reviews.,"The Cake Bake Shop 8"" Round Chocolate Cake (16...",$,3 Layers of French Valrhona Chocolate Cake M...,"Due to the perishable nature of this item, ord..."


## **ISSUE #1: Fix Column Names + Data Types**
### Convert to snake_case to improve consistency and readability:
* Lowercase
* Remove leading/trailing whitespace
* Remove parentheses
* Replace dashes/spaces with underscores

In [34]:
# Standardize column names
df.columns = (df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("(", "")
    .str.replace(")", "")
)

# Rename "rating" column to "average_rating"
df = df.rename(columns={'rating': 'average_rating'})

# Inspect columns
df.columns

Index(['sub_category', 'price', 'discount', 'average_rating', 'title',
       'currency', 'feature', 'product_description'],
      dtype='object')

## **ISSUE #2: Fix Duplicate Values**
### Remove duplicate values to ensure accurate data.

In [35]:
# Count duplicate values
print('Duplicates Count: ', df.duplicated().sum())

Duplicates Count:  3


In [36]:
# Remove duplicate values
df = df.drop_duplicates()

# Make sure duplicates were removed
print('Duplicates Count: ', df.duplicated().sum())

Duplicates Count:  0


## **ISSUE #3: Remove `NULL` Values**
### A few values in the **`price`** column are **`NULL`**. These rows will be dropped, since **`price`** is important for this analysis.

In [37]:
# Count null values in "price" columns
print('Null Values in Price Column Count: ', df['price'].isnull().sum())

Null Values in Price Column Count:  3


In [38]:
# Drop rows where "price" is null
df = df.dropna(subset=['price'])

# Make sure null rows in "price" column were removed
print('Null Values in Price Column Count: ', df['price'].isnull().sum())

Null Values in Price Column Count:  0


## **ISSUE #4: Remove `currency` Column**
### Every value in the **`currency`** column is either **`$`** or **`NULL`**. This column will be dropped, since it brings no analytical insight into the data.

In [39]:
# Make sure "currency" column has only one unique value
print('Unique Currency Values: ', df['currency'].unique())

Unique Currency Values:  ['$' nan]


In [40]:
# Drop "currency" column
df = df.drop(columns=['currency'])

# Make sure "currency" column was removed
print(df.columns)

Index(['sub_category', 'price', 'discount', 'average_rating', 'title',
       'feature', 'product_description'],
      dtype='object')


## **ISSUE #5: Fix `discount`, `average_rating`, and `title` Values**
### Some values in the **`discount`**, **`average_rating`**, and **`title`** columns have unnecessary information. These values essentially represent missing data, so they will be replaced with blank columns:
* **`discount`** → Replace **`This item is not returnable.`**, **`Limit 1 Per Member`**, **`Limit 5 Per Member`**
* **`average_rating`** → Replace **`No Reviews`**
* **`title`** → Replace **`Page Not Found!`**

In [41]:
# Find unnecessary values in "discount" column
print(df['discount'].unique())

['No Discount' 'After $30 OFF' 'After $5 OFF'
 'This item is not returnable.' 'After $20 OFF' 'After $4.50 OFF'
 'After $4 OFF' 'After $2.80 OFF' 'After $8 OFF' 'After $3.60 OFF'
 'After $3 OFF' 'After $1.50 OFF' 'After $3.30 OFF' 'After $6 OFF'
 'After $2.40 OFF' 'After $2.20 OFF' 'After $12 OFF' 'After $3.10 OFF'
 'After $5.60 OFF' 'After $2.70 OFF' 'After $10 OFF' 'After $9.30 OFF'
 'After $50 OFF' 'After $2.50 OFF' 'After $70 OFF' 'After $60 OFF'
 'After $40 - $80 OFF' 'After $40 - $70 OFF' 'After $40 OFF'
 'After $80 OFF' 'After $3.50 OFF' 'Limit 1 Per Member'
 'Limit 5 Per Member' 'After $6.50 OFF' 'After $3.80 OFF'
 'After $2.60 OFF' 'After $7 OFF' 'After $4.10 OFF' 'After $2 OFF'
 'After $2.30 OFF']


In [42]:
# Replace unnecessary "discount" data with blank cell
df["discount"] = df["discount"].replace([
    "This item is not returnable.", 
    "Limit 1 Per Member", 
    "Limit 5 Per Member"
    ],
    ""
)

# Replace missing "average_rating" data with blank cell
df["average_rating"] = df["average_rating"].replace(
    "No Reviews",
    ""
)

# Replace missing "title" data with blank cell
df["title"] = df["title"].replace(
    "Page Not Found!",
    ""
)

## **ISSUE #6: Fix `discount` Values**
### All values in the **`discount`** column are written as descriptive text, and some contain a range of prices rather than a single discount value. Only the numerical discount will be extracted and preserved, in order to enhance querying and visualizations:
* **EX:** **`After $4.50 OFF`** → **`4.50`**
* **EX:** **`After $40 - $80 OFF`** → Find average between ranges → **`60`**
* **`No Discount`** → **`0`**

In [43]:
# Define function to clean "discount" column
def clean_discount(value):
    
    if pd.isna(value):
        return np.nan

    # Replace "No Discount" with "0"
    if value.strip() == "No Discount":
        return 0.0

    # Extract numerical discount from string
    numbers = re.findall(r'\d+\.?\d*', value)

    if not numbers:
        return np.nan

    numbers = [float(num) for num in numbers]

    # If discount is a range between two numbers, extract its average
    if len(numbers) > 1:
        return sum(numbers) / len(numbers)

    # Return numerical discount value
    return numbers[0]

# Apply cleaning function
df['discount'] = df['discount'].apply(clean_discount)

# Replace empty fields with "0", to prevent errors during querying
df['discount'] = df['discount'].fillna(0)

# Make sure only the numerical discount was preserved
print(df['discount'].describe())

count    1751.000000
mean        0.822045
std         5.514765
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        80.000000
Name: discount, dtype: float64


## **ISSUE #7: Fix `price` Values**
### Some values in the **`price`** column contain a range between two numbers, such as **`$32.99through-$83.99`**, rather than a single value. Only the average between the two prices, such as **`58.49`**, will be extracted and preserved, in order to enhance querying and visualizations:

In [44]:
# Define function to clean "price" column
def clean_price(value):
    
    if pd.isna(value):
        return np.nan

    value = str(value)

    # Extract numerical price from string
    if "through" in value.lower():
        numbers = re.findall(r'\d[\d,]*\.?\d*', value)
        numbers = [float(num.replace(',', '')) for num in numbers]

        # If price is a range between two numbers, extract its average
        if len(numbers) == 2:
            return round(sum(numbers) / 2, 2)

    # Make sure prices > 999.99 aren't mistakenly altered
    match = re.search(r'\d[\d,]*\.?\d*', value)
    if match:
        return round(float(match.group().replace(',', '')), 2)

    return np.nan

# Apply cleaning function
df['price'] = df['price'].apply(clean_price)

# Make sure only the numerical price was preserved, and prices > 999.99 weren't altered
print(df['price'].describe())

count    1751.000000
mean       61.369874
std       121.486145
min         3.990000
25%        14.990000
50%        23.990000
75%        52.990000
max      1999.990000
Name: price, dtype: float64


## **ISSUE #8: Fix `average_rating` Values**
### All values in the **`average_rating`** column are written as descriptive text, such as **`Rated 4.3 out of 5 stars based on 265 reviews`**. Only the numerical rating, such as **`4.3`**, will be extracted and preserved, in order to enhance querying and visualizations.

In [45]:
# Extract numerical rating from string
df['average_rating'] = (
    df['average_rating']
    .str.extract(r'(\d+\.?\d*)')
    .astype(float)
)

# Replace empty fields with "0", to prevent errors during querying
df['average_rating'] = df['average_rating'].fillna(0)

# Make sure only the numerical rating was preserved
print(df['average_rating'].describe())

count    1751.000000
mean        1.524272
std         2.094201
min         0.000000
25%         0.000000
50%         0.000000
75%         4.200000
max         5.000000
Name: average_rating, dtype: float64


## **ISSUE #9: Fix `title`, `feature`, and `product_description` Values**
### Most values in the **`title`**, **`feature`**, and **`product_description`** columns have quotes **`""`**, commas **`,`**, newline markers **`\n`**, non-ASCII characters **`è, â, í, õ, ü`**, and other inputs that may cause parsing issues. These will be removed, while the descriptive text will be preserved, in order to prevent parsing errors.

In [46]:
# Define function to clean text columns
def clean_text(value):
    if pd.isna(value):
        return value

    value = str(value)

    # Replace newline characters with a space
    value = value.replace('\n', ' ').replace('\\n', ' ')

    # Remove non-ASCII characters
    value = value.encode('ascii', errors='ignore').decode('ascii')

    # Replace punctuation with a space
    value = re.sub(r'[^\w\s]', ' ', value)
    
    # Remove any remaining special characters
    value = re.sub(r'[^a-zA-Z0-9\s]', '', value)
    
    # Remove double spaces
    value = re.sub(r'\s+', ' ', value).strip()

    return value

# Apply cleaning function to text columns
text_columns = ['title', 'feature', 'product_description']

for col in text_columns:
    df[col] = df[col].apply(clean_text)

# Make sure text columns were cleaned
print(df[text_columns].head())

                                               title  \
0  Davids Cookies Mile High Peanut Butter Cake 6 ...   
1  The Cake Bake Shop 8 Round Carrot Cake 16 22 S...   
2  St Michel Madeleine Classic French Sponge Cake...   
3  David s Cookies Butter Pecan Meltaways 32 oz 2...   
4  Davids Cookies Premier Chocolate Cake 7 2 lbs ...   

                                             feature  \
0  10 Peanut Butter Cake Certified Kosher OU D 14...   
1  Spiced Carrot Cake with Cream Cheese Frosting ...   
2  100 count Individually wrapped Made in and Imp...   
3  Butter Pecan Meltaways 32 oz 2 Pack No Preserv...   
4  10 Four Layer Chocolate Cake Certified Kosher ...   

                                 product_description  
0  A cake the dessert epicure will die for Our To...  
1  Due to the perishable nature of this item orde...  
2  Moist and buttery sponge cakes with the tradit...  
3  These delectable butter pecan meltaways are th...  
4  A cake the dessert epicure will die for To the..

## **ISSUE #10: Feature Engineer Keyword Flag Columns**
### Most values in the **`title`**, **`feature`**, and **`product_description`** columns contain long text fields, with certain keywords that would be useful for data analysis. First, find the most frequently occurring words in all three columns. Use this information to decide the best keywords to flag.

In [47]:
# Combine "title", "feature", and "product_description" columns into one
df['combined_text'] = (
    df['title'].fillna('') + ' ' +
    df['feature'].fillna('') + ' ' +
    df['product_description'].fillna('')
)

# Replace any remaining empty fields with "", to prevent errors during parsing
df = df.replace(np.nan, "")

# Tokenize words in "combined_text" column
words = df['combined_text'].str.lower().str.findall(r'\b[a-z]+\b')

all_words = [word for sublist in words for word in sublist]

# Remove common stopwords that block useful keywords from the top of the list
stopwords = set([
    'and', 'or', 'with', 'for', 'the', 'to', 'of', 'in', 'on', 'at', 'available', 'features',
    'this', 'that', 'is', 'are', 'includes', 'include', 'a', 'your', 'arrival', 'bar', 'every',
    'from', 'you', 'our', 'no', 'total', 'be', 's', 'made', 'these', 'arrive', 'individually',
    'it', 'per', 'can', 'will', 'not', 'we', 'all', 'as', 'use', 'an', 'white', 'included',
    'by', 'day', 'x', 'kirkland', 'signature', 'only', 'its', 'contains', 'black', 'checkout',
    'has', 'them', 'may', 'best', 'before', 'days', 'product', 'net', 'through', 'enjoy', 'have',
    'just', 'any', 'each', 'orders', 'oz', 'box', 'bag', 'bags', 'fl', 'delicious', 'hand',
    'pack', 'packs', 'lb', 'lbs', 'date', 'one', 'size', 'up', 'during', 'great', 'dark',
    'count', 'weight', 'more', 'order', 'after', 'cup', 'ship', 'help', 'taste', 'flavors'
])

filtered_words = [w for w in all_words if w not in stopwords]

# Count frequency of words in "combined_text" column
word_counts = Counter(filtered_words)

# View top 50 most frequent words in "combined_text" column
word_counts.most_common(50)

[('free', 1129),
 ('chocolate', 1063),
 ('flowers', 604),
 ('coffee', 561),
 ('flavor', 551),
 ('kosher', 533),
 ('water', 525),
 ('organic', 512),
 ('certified', 481),
 ('beef', 458),
 ('perfect', 436),
 ('quality', 422),
 ('frozen', 417),
 ('gift', 411),
 ('fresh', 403),
 ('gluten', 394),
 ('usa', 385),
 ('milk', 380),
 ('sugar', 366),
 ('variety', 354),
 ('non', 352),
 ('natural', 335),
 ('ingredients', 323),
 ('sweet', 322),
 ('candy', 322),
 ('whole', 317),
 ('premium', 310),
 ('wagyu', 306),
 ('food', 296),
 ('caviar', 291),
 ('steak', 287),
 ('roast', 277),
 ('usda', 275),
 ('original', 256),
 ('chicken', 256),
 ('high', 250),
 ('rich', 247),
 ('blend', 245),
 ('butter', 244),
 ('full', 242),
 ('added', 241),
 ('salt', 236),
 ('protein', 236),
 ('wild', 234),
 ('snack', 229),
 ('artificial', 223),
 ('cookies', 221),
 ('fat', 220),
 ('gmo', 219),
 ('steaks', 216)]

### Second, create binary columns for each important product attribute. This will flag whether each product **`is_clean_label`**, **`is_eco_friendly`**, **`is_health_conscious`**, **`is_high_quality`**, **`is_plant_based`**, **`is_shelf_stable`**, or **`has_religious_certification`**. These new columns extract features from long text fields that are useful for querying and uncovering trends.

In [48]:
# Name new binary columns, and assign appropriate keywords to each
keyword_groups = {
    'is_clean_label': ['organic', 'non gmo', 'all natural', 'no artificial', 'no additives', 'no added additives', 'no synthetic',
                       'no preservatives', 'no added preservatives', 'antibiotic free', 'no antibiotics', 'without antibiotics', 'usda',
                       'pesticide', 'pesticides', 'herbicide', 'herbicides', 'hormone', 'hormones', 'natural ingredients', 'chemical free'],
    
    'is_eco_friendly': ['eco friendly', 'sustainable', 'sustainably', 'sustainability', 'biodegradable', 'compostable', 'reusable',
                        'recycle', 'recycled', 'recyclable', 'plastic free', 'ethically sourced', 'zero waste', 'renewable'],
    
    'is_health_conscious': ['keto', 'paleo', 'low sugar', 'no sugar', '0g sugar', '0g sugars', 'zero sugar', 'no sugars', 'healthy gut',
                            '0g trans fat', 'trans fat free', 'low fat', 'no fat', 'low carb', 'no carb', 'high in fiber', 'antioxidants', 
                            'zero calorie', 'zero calories', 'no calorie', 'no calories', 'low calorie', 'low calories', 'healthy',
                            'low cholesterol', 'cholesterol free', 'no cholesterol', 'high protein', 'high in protein',
                            'low sodium', 'low in sodium', 'fat free', 'no added sugar', 'sugar free', 'heart healthy',
                            'low glycemic', 'good source of', 'excellent source of', '0g saturated fat', '0g sat fat'],
    
    'is_high_quality': ['high quality', 'artisan', 'artisanal', 'premium', 'luxury', 'luxurious',
                        'specialty', 'handcrafted', 'hand crafted', 'handmade', 'hand made'],
    
    'is_plant_based': ['plant based', 'vegan', 'vegetarian friendly', 'dairy free', 'non dairy', 'free of dairy', 'cruelty free'],
    
    'is_shelf_stable': ['shelf stable', 'shelf life', 'pantry', 'no refrigeration', 
                        'dried', 'canned', 'ready to eat', 'ready to serve'],
    
    'has_religious_certification': ['halal', 'certifiedhalal', 'kosherhalal', 'kosher',
                                    'koshercertified', 'certifiedkosher', 'verifiedkosher']
}

# Flag products in "combined_text" column with designated keywords
for col, keywords in keyword_groups.items():
    df[col] = df['combined_text'].str.lower().apply(
        lambda x: int(any(k in x for k in keywords))
    )

# Drop "combined_text" column, since no longer needed
df = df.drop(columns=['combined_text'])

## **View Cleaned Dataset**

In [49]:
# Inspect dataset
print("Dataset Dimensions: ", df.shape)
print("Dataset Attributes: ", df.columns.values)

df.info()
df.head(10)

Dataset Dimensions:  (1751, 14)
Dataset Attributes:  ['sub_category' 'price' 'discount' 'average_rating' 'title' 'feature'
 'product_description' 'is_clean_label' 'is_eco_friendly'
 'is_health_conscious' 'is_high_quality' 'is_plant_based'
 'is_shelf_stable' 'has_religious_certification']
<class 'pandas.core.frame.DataFrame'>
Index: 1751 entries, 0 to 1756
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   sub_category                 1751 non-null   object 
 1   price                        1751 non-null   float64
 2   discount                     1751 non-null   float64
 3   average_rating               1751 non-null   float64
 4   title                        1751 non-null   object 
 5   feature                      1751 non-null   object 
 6   product_description          1751 non-null   object 
 7   is_clean_label               1751 non-null   int64  
 8   is_eco_friendly         

,sub_category,price,discount,average_rating,title,feature,product_description,is_clean_label,is_eco_friendly,is_health_conscious,is_high_quality,is_plant_based,is_shelf_stable,has_religious_certification
0,Bakery & Desserts,56.99,0.0,4.3,Davids Cookies Mile High Peanut Butter Cake 6 ...,10 Peanut Butter Cake Certified Kosher OU D 14...,A cake the dessert epicure will die for Our To...,1,0,0,0,0,0,1
1,Bakery & Desserts,159.99,0.0,5.0,The Cake Bake Shop 8 Round Carrot Cake 16 22 S...,Spiced Carrot Cake with Cream Cheese Frosting ...,Due to the perishable nature of this item orde...,0,0,0,0,0,0,0
2,Bakery & Desserts,44.99,0.0,4.1,St Michel Madeleine Classic French Sponge Cake...,100 count Individually wrapped Made in and Imp...,Moist and buttery sponge cakes with the tradit...,1,0,0,0,0,0,0
3,Bakery & Desserts,39.99,0.0,4.7,David s Cookies Butter Pecan Meltaways 32 oz 2...,Butter Pecan Meltaways 32 oz 2 Pack No Preserv...,These delectable butter pecan meltaways are th...,1,0,0,0,0,0,1
4,Bakery & Desserts,59.99,0.0,4.5,Davids Cookies Premier Chocolate Cake 7 2 lbs ...,10 Four Layer Chocolate Cake Certified Kosher ...,A cake the dessert epicure will die for To the...,1,0,0,0,0,0,1
5,Bakery & Desserts,59.99,0.0,4.4,David s Cookies Mango Strawberry Cheesecake 2 ...,2 count Strawberry Cheesecake Mango Cheesecake...,Strawberry Cheesecake There s only one way to ...,0,0,0,0,0,0,1
6,Bakery & Desserts,74.99,0.0,4.7,La Grande Galette French Butter Cookies 1 3 lb...,1 3 lb 6 count Baked in and Imported from Fran...,Once upon a time in the French coastal town of...,1,0,0,1,0,0,0
7,Bakery & Desserts,59.99,0.0,4.4,David s Cookies No Sugar Added Cheesecake Marb...,2 count No Sugar Added Kosher OU Dairy,Creamy Dreamy This smooth creamy cheesecake ha...,0,0,1,0,0,0,1
8,Bakery & Desserts,29.99,0.0,4.4,David s Cookies Brownie and Cookie Combo Pack,6 Rocky Road Brownies 12 Chocoloate Chunk Cook...,Due to the perishable nature of this product o...,0,0,0,0,0,0,1
9,Bakery & Desserts,159.99,0.0,5.0,The Cake Bake Shop 8 Round Chocolate Cake 16 2...,3 Layers of French Valrhona Chocolate Cake Mou...,Due to the perishable nature of this item orde...,0,0,0,0,0,0,0


## **Save Cleaned Dataset**

In [50]:
# Save cleaned dataset
df.to_csv("grocery_store_dataset_cleaned.csv", index=False)